# 06 — Modélisation

**Objectif** : prédire le nombre de ventes du mois suivant pour chaque couple
région × plage de prix, puis enregistrer le meilleur modèle.

Le découpage est **chronologique**. La référence à battre est la persistance
(le mois suivant égale le mois courant).

In [ ]:
import importlib
import sys
from pathlib import Path

racine = Path.cwd().resolve()
if not (racine / "src").is_dir():
    racine = racine.parent
sys.path.insert(0, str(racine))

import src

importlib.reload(src)


import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src import FEATURES_CSV, FIGURES_DIR, MODELS_DIR

sns.set_theme(style="whitegrid")
ALEA = 42
IDENTIFIANTS = {"periode", "id_region", "region", "code_plage_prix", "plage_prix"}

In [ ]:
features = pd.read_csv(FEATURES_CSV, parse_dates=["periode"]).sort_values("periode")

CIBLE = "cible_ventes_suivantes"
EXPLICATIVES = [c for c in features.columns if c not in IDENTIFIANTS | {CIBLE}]

print(f"{len(features)} observations, {len(EXPLICATIVES)} variables")
print(f"Couverture : {features['periode'].min():%Y-%m} → {features['periode'].max():%Y-%m}")

## Découpage chronologique

La coupure porte sur les mois, pas sur les lignes.

In [ ]:
periodes = np.sort(features["periode"].unique())
coupure = periodes[int(len(periodes) * 0.8)]

entrainement = features[features["periode"] < coupure]
test = features[features["periode"] >= coupure]

X_train, y_train = entrainement[EXPLICATIVES], entrainement[CIBLE]
X_test, y_test = test[EXPLICATIVES], test[CIBLE]

print(f"Entraînement : {len(X_train)} lignes (< {pd.Timestamp(coupure):%Y-%m})")
print(f"Test         : {len(X_test)} lignes (≥ {pd.Timestamp(coupure):%Y-%m})")

In [ ]:
def evaluer(nom, y_vrai, y_prevu):
    y_vrai = np.asarray(y_vrai, dtype=float)
    y_prevu = np.asarray(y_prevu, dtype=float)
    non_nuls = y_vrai != 0
    return {
        "modele": nom,
        "mae": mean_absolute_error(y_vrai, y_prevu),
        "rmse": np.sqrt(mean_squared_error(y_vrai, y_prevu)),
        "mape_pct": np.mean(np.abs((y_vrai[non_nuls] - y_prevu[non_nuls]) / y_vrai[non_nuls])) * 100,
        "r2": r2_score(y_vrai, y_prevu),
    }


resultats = [evaluer("Persistance (ventes_retard_1)", y_test, X_test["ventes_retard_1"])]

## Modèles candidats

In [ ]:
candidats = {
    "Ridge": Pipeline(
        [
            ("imputation", SimpleImputer(strategy="median")),
            ("normalisation", StandardScaler()),
            ("regression", RidgeCV(alphas=np.logspace(-3, 3, 13))),
        ]
    ),
    "Forêt aléatoire": Pipeline(
        [
            ("imputation", SimpleImputer(strategy="median")),
            (
                "regression",
                RandomForestRegressor(
                    n_estimators=400, min_samples_leaf=2, random_state=ALEA, n_jobs=-1
                ),
            ),
        ]
    ),
    "Gradient boosting": Pipeline(
        [
            ("imputation", SimpleImputer(strategy="median")),
            (
                "regression",
                GradientBoostingRegressor(
                    n_estimators=300, learning_rate=0.05, max_depth=3, random_state=ALEA
                ),
            ),
        ]
    ),
}

In [ ]:
validation = TimeSeriesSplit(n_splits=5)
modeles_entraines = {}

for nom, pipeline in candidats.items():
    scores_cv = cross_val_score(
        pipeline, X_train, y_train, cv=validation, scoring="neg_mean_absolute_error"
    )
    pipeline.fit(X_train, y_train)
    modeles_entraines[nom] = pipeline

    mesures = evaluer(nom, y_test, pipeline.predict(X_test))
    mesures["mae_cv"] = -scores_cv.mean()
    resultats.append(mesures)
    print(f"{nom} : MAE test = {mesures['mae']:.1f} | MAE CV = {mesures['mae_cv']:.1f}")

In [ ]:
tableau = pd.DataFrame(resultats).sort_values("mae").reset_index(drop=True)
tableau.round(3)

In [ ]:
classement = tableau[tableau["modele"].isin(modeles_entraines)]
meilleur_nom = classement.iloc[0]["modele"]
meilleur_modele = modeles_entraines[meilleur_nom]

mae_reference = tableau.loc[tableau["modele"].str.startswith("Persistance"), "mae"].iloc[0]
mae_meilleur = classement.iloc[0]["mae"]
gain = (1 - mae_meilleur / mae_reference) * 100

print(f"Modèle retenu : {meilleur_nom}")
print(f"MAE {mae_meilleur:.1f} contre {mae_reference:.1f} pour la persistance ({gain:+.1f} %)")
if gain <= 0:
    print("Attention : aucun modèle ne bat la persistance.")

In [ ]:
regression = meilleur_modele.named_steps["regression"]
poids = getattr(regression, "feature_importances_", None)
if poids is None:
    poids = np.abs(getattr(regression, "coef_", np.zeros(len(EXPLICATIVES))))

importances = pd.Series(poids, index=EXPLICATIVES).sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 6))
sns.barplot(x=importances.to_numpy(), y=importances.index, color="#2a6f97")
plt.title(f"Variables les plus influentes — {meilleur_nom}")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_importance_variables.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
comparaison = test[["periode"]].copy()
comparaison["observe"] = y_test.to_numpy()
comparaison["prevu"] = meilleur_modele.predict(X_test)
agrege = comparaison.groupby("periode")[["observe", "prevu"]].sum()

plt.figure(figsize=(12, 5))
plt.plot(agrege.index, agrege["observe"], label="Observé", marker="o")
plt.plot(agrege.index, agrege["prevu"], label="Prévu", marker="s", linestyle="--")
plt.title("Ventes québécoises observées et prévues (période de test)")
plt.ylabel("Nombre de ventes")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_observe_vs_prevu.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

artefact = {
    "modele": meilleur_modele,
    "nom": meilleur_nom,
    "explicatives": EXPLICATIVES,
    "cible": CIBLE,
    "periode_coupure": pd.Timestamp(coupure),
    "metriques_test": classement.iloc[0].to_dict(),
}

chemin = MODELS_DIR / "best_model.pkl"
joblib.dump(artefact, chemin)
print(f"Écrit : {chemin}")